In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [2]:
column_names = [
    "CrimeRate",  # The per capita crime rate by town. lower housing values
    "ResidentialLandZoned", # The proportion of residential land zoned for lots over 25,000 sq.ft.
    "IndustrialLandUse", # The proportion of non-retail business acres per town.
    "CharlesRiver", # Charles River dummy variable (1 if tract bounds river; 0 otherwise)
    "NitrixOxide", # Nitric oxides concentration (parts per 10 million)
    "AvgRoomPerDwelling", # The average number of rooms per dwelling
    "HousingAge", # The proportion of owner-occupied units built prior to 1940
    "DistanceToWork", # Weighted distances to five Boston employment centres
    "HighwayAccess", # Index of accessibility to radial highways
    "PropertyTaxRate", # Full-value property-tax rate per $10,000
    "Pupil-TeacherRatio", # The pupil-teacher ratio by town
    "B", # 1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town
    "LowSocioEcomic", # The percentage of lower status of the population
    "Value", # Median value of owner-occupied homes in $1000s
]
df = pd.read_csv("data/housing.csv", sep=r"\s+", header=None, names=column_names)

In [3]:
df.head(2)

,CrimeRate,ResidentialLandZoned,IndustrialLandUse,CharlesRiver,NitrixOxide,AvgRoomPerDwelling,HousingAge,DistanceToWork,HighwayAccess,PropertyTaxRate,Pupil-TeacherRatio,B,LowSocioEcomic,Value
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.9,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.9,9.14,21.6


In [2]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import mlflow

In [3]:
mlflow.set_tracking_uri("http://localhost:5020")
mlflow.set_experiment("Boston Model Experiment")
mlflow.set_registry_uri("sqlite:///mlflow_registry.db")


In [6]:
feature_cols = ["CrimeRate", "ResidentialLandZoned", "IndustrialLandUse", "CharlesRiver", "NitrixOxide", "AvgRoomPerDwelling", "HousingAge", "DistanceToWork", "HighwayAccess", "PropertyTaxRate", "Pupil-TeacherRatio", "B", "LowSocioEcomic"]
X = df[feature_cols]
y = df['Value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
mlflow.start_run(run_name="XGBRegressor Run")

xgb_model = XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5

mlflow.log_metric("rmse", rmse)
mlflow.xgboost.log_model(xgb_model, "xgbr_model")

model_name = "XGBRegressor Model"
model_uri = f"runs:/{mlflow.active_run().info.run_id}/xgbr_model"
mlflow.register_model(model_uri, model_name)

mlflow.end_run()



2025/09/04 23:09:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\ProgramData\anaconda3\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:09:22] WARNING: C:\b\abs_90_bwj_86a\croot\xgboost-split_1724073762025\work\src\c_api\c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2025/09/04 23:09:27 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/09/04 23:09:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/09/04 23:09:30 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Running upgrade  -> 451

🏃 View run XGBRegressor Run at: http://localhost:5020/#/experiments/417350048653659120/runs/c17cc64c833140929519e5e7ab80e20b
🧪 View experiment at: http://localhost:5020/#/experiments/417350048653659120


1. A logged model is a model artifact saved during an MLflow run. It is stored in the MLflow tracking server or a specified artifact storage location. Logged models are primarily used for experimentation and tracking purposes.
- Purpose: Captures the model, its parameters, metrics, and artifacts during training.
- Storage: Saved as part of an MLflow run under the artifacts section.
- Access: Can be loaded using the mlflow.<model_flavor>.load_model() method with a run-relative path (e.g., runs:/<run_id>/<model_path>).
- Use Case: Ideal for experimentation, debugging, and intermediate model evaluation.

2. A registered model is a model that has been added to the MLflow Model Registry. It provides a centralized system for managing models across their lifecycle, including versioning, staging, and deployment.

- Purpose: Enables collaborative model management with features like versioning, aliases, and metadata tagging.
- Storage: Stored in the Model Registry with a unique name and associated versions.
- Access: Can be loaded using a model URI (e.g., models:/<model_name>/<version> or models:/<model_name>@<alias>).
- Use Case: Suitable for production workflows, deployment, and governance.

In [ ]:
# Load registered model and make predictions
loaded_model = mlflow.pyfunc.load_model(model_uri)
loaded_model.predict(X_test)

In [7]:
for expr in mlflow.search_experiments(filter_string="Boston Model Experiment"):
    print(expr)

RestException: INVALID_PARAMETER_VALUE: Invalid clause(s) in filter string: Boston Model, Experiment